# Fox Audio Detection — Complete Reproducible Pipeline

**TECHIN 513A Final Project**

Binary audio classification: Detect whether an audio clip contains a **red fox (*Vulpes vulpes*)** vocalisation.

## Pipeline Overview

| Step | Description |
|------|-------------|
| 0 | Environment setup & dependency installation |
| 1 | Download audio data from Xeno-canto |
| 2 | Segment recordings into 3-second clips |
| 3 | Data exploration & visualisation |
| 4 | Feature extraction (MFCCs + spectrograms) |
| 5 | Train baseline models (SVM / RF / GB) |
| 6 | Train CNN (EfficientNet-B0) |
| 7 | Evaluate both models on test set |
| 8 | Training-set evaluation (overfitting check) |
| 9 | Model comparison |
| 10 | Interactive Gradio demo |

---

## Step 0 — Environment Setup

Install all required packages and create the project directory structure.

> **Note:** On first run, this cell installs dependencies. Restart the kernel after installation if prompted.

In [ ]:
# ── 0a. Install dependencies ──────────────────────────────────────────────
import subprocess, sys

REQUIRED = [
    "librosa>=0.10.0", "soundfile", "numpy", "pandas",
    "scikit-learn", "matplotlib", "seaborn",
    "torch>=2.0.0", "torchvision>=0.15.0", "torchaudio",
    "Pillow", "tqdm", "gradio>=4.0.0", "requests",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + REQUIRED
)
print("All dependencies installed.")

In [ ]:
# ── 0b. Create project directory structure ────────────────────────────────
import os, sys

# Determine project root (the fox_detection directory)
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))  # where this .ipynb lives
PROJECT_ROOT = os.path.join(NOTEBOOK_DIR, "fox_detection")

DIRECTORIES = [
    "data/raw/fox", "data/raw/nonfox",
    "data/clips/fox", "data/clips/nonfox",
    "data/features",
    "data/spectrograms/fox", "data/spectrograms/nonfox",
    "models/baseline", "models/cnn",
    "src",
]

for d in DIRECTORIES:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)

# Ensure src/ is importable
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print("Directory structure created.")

In [ ]:
# ── 0c. Common imports & constants ────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
%matplotlib inline

# SSL fix for macOS (needed for torchvision model downloads)
import certifi, os
os.environ["SSL_CERT_FILE"] = certifi.where()

# ── Path constants ────────────────────────────────────────────────────
DATA_DIR      = os.path.join(PROJECT_ROOT, "data")
RAW_FOX       = os.path.join(DATA_DIR, "raw", "fox")
RAW_NONFOX    = os.path.join(DATA_DIR, "raw", "nonfox")
CLIPS_DIR     = os.path.join(DATA_DIR, "clips")
MANIFEST_CSV  = os.path.join(DATA_DIR, "manifest.csv")
FEATURE_DIR   = os.path.join(DATA_DIR, "features")
SPEC_DIR      = os.path.join(DATA_DIR, "spectrograms")
BASELINE_DIR  = os.path.join(PROJECT_ROOT, "models", "baseline")
CNN_DIR       = os.path.join(PROJECT_ROOT, "models", "cnn")
BASELINE_PATH = os.path.join(BASELINE_DIR, "model.pkl")
CNN_PATH      = os.path.join(CNN_DIR, "best.pt")

# Audio constants
SR = 22050
CLIP_DURATION = 3.0
OVERLAP = 0.5
N_MFCC = 40

# Device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"PyTorch device: {DEVICE}")
print(f"NumPy:   {np.__version__}")
print(f"PyTorch: {torch.__version__}")

---
## Step 1 — Download Audio Data from Xeno-canto

We download fox (*Vulpes vulpes*) vocalisations and non-fox sounds (birds + other mammals) from the [Xeno-canto](https://xeno-canto.org/) API v3.

| Category | Queries | Destination |
|----------|---------|-------------|
| Fox (quality A+B) | `gen:vulpes sp:vulpes q:A/B` | `data/raw/fox/` |
| Non-fox birds | 5 European species (blackbird, robin, great tit, chaffinch, crow) | `data/raw/nonfox/` |
| Non-fox mammals | Grey wolf, red deer | `data/raw/nonfox/` |

> **API key:** Set `XC_KEY` below. Already-downloaded files are skipped automatically.

In [ ]:
from src.download_data import download_data

# ─── Set your Xeno-canto API key here ─────────────────────────────────
XC_KEY = "b40e7a6e9c8dda5130b90d79405ed7c6fcaaa1f5"

fox_new, nonfox_new = download_data(
    xc_key=XC_KEY,
    fox_dir=RAW_FOX,
    nonfox_dir=RAW_NONFOX,
)

n_fox = len([f for f in os.listdir(RAW_FOX) if f.endswith(".mp3")])
n_nonfox = len([f for f in os.listdir(RAW_NONFOX) if f.endswith(".mp3")])
print(f"\nTotal recordings: {n_fox} fox, {n_nonfox} non-fox")

---
## Step 2 — Audio Segmentation

Segment each recording into overlapping 3-second clips (hop = 2.5 s) and build a manifest CSV.

| Parameter | Value |
|-----------|-------|
| Clip duration | 3.0 s |
| Overlap | 0.5 s |
| Sample rate | 22,050 Hz |
| Method | Fixed-length with zero-pad tail |

In [ ]:
from src.segmentation import build_manifest

manifest_df = build_manifest(
    fox_dir=RAW_FOX,
    nonfox_dir=RAW_NONFOX,
    output_dir=CLIPS_DIR,
    manifest_path=MANIFEST_CSV,
    sr=SR,
    clip_duration=CLIP_DURATION,
    overlap=OVERLAP,
    method="fixed",
)

print(f"\nManifest: {len(manifest_df)} clips")
print(manifest_df["label"].value_counts())
manifest_df.head()

---
## Step 3 — Data Exploration

Visualise class balance, sample waveforms, and spectrograms.

In [ ]:
from src.audio_utils import load_audio, normalise_audio, compute_log_mel_spectrogram

# ── 3a. Class distribution ────────────────────────────────────────────
df = pd.read_csv(MANIFEST_CSV)
print(f"Total clips: {len(df):,}")
print(f"\nClass distribution:")
print(df["label"].value_counts())

fig, ax = plt.subplots(figsize=(5, 3))
df["label"].value_counts().plot.bar(ax=ax, color=["steelblue", "coral"])
ax.set_title("Class Distribution")
ax.set_ylabel("Number of clips")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3b. Sample waveforms & spectrograms ───────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for col, label in enumerate(["fox", "nonfox"]):
    subset = df[df["label"] == label].sample(2, random_state=42)
    for row_offset, (_, clip_row) in enumerate(subset.iterrows()):
        clip_path = os.path.join(DATA_DIR, clip_row["clip_path"])
        y, sr = load_audio(clip_path, target_sr=SR)
        y = normalise_audio(y)
        spec = compute_log_mel_spectrogram(y, sr)
        
        c = col * 2 + row_offset
        # Waveform
        t = np.arange(len(y)) / sr
        axes[0, c].plot(t, y, linewidth=0.4, color="steelblue")
        axes[0, c].set_title(f"{label} #{row_offset+1}", fontsize=10)
        axes[0, c].set_xlabel("Time (s)")
        axes[0, c].set_ylabel("Amplitude")
        
        # Spectrogram
        axes[1, c].imshow(spec, aspect="auto", origin="lower", cmap="magma")
        axes[1, c].set_title(f"Log-Mel Spectrogram", fontsize=10)
        axes[1, c].set_xlabel("Time frames")
        axes[1, c].set_ylabel("Mel band")

fig.suptitle("Sample Waveforms & Spectrograms", fontsize=14)
fig.tight_layout()
plt.show()

---
## Step 4 — Feature Extraction

Extract two types of features from the segmented clips:

1. **MFCC feature vectors** (40 coefficients → 80-dim mean+std) for the baseline classifiers
2. **Log-mel spectrogram PNG images** (128 mel bands) for the CNN

Both extractors cache results and skip already-processed clips on re-runs.

In [ ]:
from src.features import extract_mfcc_dataset, extract_spectrogram_dataset

# 4a. MFCC features (for baseline)
print("Extracting MFCC features...")
X_mfcc, y_mfcc = extract_mfcc_dataset(
    MANIFEST_CSV, feature_dir=FEATURE_DIR, sr=SR, n_mfcc=N_MFCC
)
print(f"MFCC: X.shape = {X_mfcc.shape}, y.shape = {y_mfcc.shape}")
print(f"  fox: {(y_mfcc == 1).sum()}, nonfox: {(y_mfcc == 0).sum()}")

# 4b. Spectrogram images (for CNN)
print("\nExtracting spectrogram images...")
extract_spectrogram_dataset(
    MANIFEST_CSV, spec_dir=SPEC_DIR, sr=SR
)

n_fox_specs = len(os.listdir(os.path.join(SPEC_DIR, "fox")))
n_nonfox_specs = len(os.listdir(os.path.join(SPEC_DIR, "nonfox")))
print(f"Spectrograms: {n_fox_specs} fox, {n_nonfox_specs} nonfox")

---
## Step 5 — Train Baseline Models

Train three scikit-learn classifiers on the MFCC feature vectors:

| Model | Configuration |
|-------|---------------|
| **SVM** | RBF kernel, `probability=True`, StandardScaler |
| **Random Forest** | 200 trees, `n_jobs=-1` |
| **Gradient Boosting** | 200 estimators |

Data split: **70% train / 15% validation / 15% test** (stratified, `random_state=42`).
The best model by validation F1 is saved to `models/baseline/model.pkl`.

In [ ]:
from src.baseline_model import train_baseline

best_clf = train_baseline(
    manifest_csv=MANIFEST_CSV,
    feature_dir=FEATURE_DIR,
    model_dir=BASELINE_DIR,
    sr=SR,
    n_mfcc=N_MFCC,
)

print(f"\nBest baseline model: {best_clf.model_type}")
print(f"Saved to: {BASELINE_PATH}")

---
## Step 6 — Train CNN (EfficientNet-B0)

Fine-tune a pre-trained EfficientNet-B0 on the spectrogram images.

| Hyperparameter | Value |
|----------------|-------|
| Backbone | EfficientNet-B0 (ImageNet pre-trained) |
| Classifier head | Dropout(0.3) → Linear(1280, 2) |
| Optimiser | AdamW (lr=1e-3, weight_decay=1e-4) |
| Scheduler | CosineAnnealingLR |
| Loss | Weighted CrossEntropyLoss (inverse class freq) |
| Early stopping | Patience = 7 (by val F1) |
| Augmentation | HFlip, FreqMask(20), TimeMask(20), GaussNoise(0.01) |
| Image size | 128 x 128 |

In [ ]:
from src.train_cnn import train_cnn

best_ckpt_path = train_cnn(
    manifest_csv=MANIFEST_CSV,
    spec_dir=SPEC_DIR,
    model_dir=CNN_DIR,
    backbone="efficientnet_b0",
    pretrained=True,
    epochs=30,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    dropout=0.3,
    patience=7,
    img_size=(128, 128),
    num_workers=0,
    device=DEVICE,
)

print(f"\nBest checkpoint: {best_ckpt_path}")

# Show training curves
curves_path = os.path.join(CNN_DIR, "training_curves.png")
if os.path.exists(curves_path):
    from PIL import Image
    img = Image.open(curves_path)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Training Curves (Loss & Validation F1)")
    plt.tight_layout()
    plt.show()

---
## Step 7 — Evaluate Both Models on Test Set

Run inference on the held-out **test split** (15%) for both the baseline and CNN models.
Generate confusion matrices, precision-recall curves, and ROC curves.

In [ ]:
from src.evaluate import evaluate_model, compare_models

EVAL_DIR = os.path.join(PROJECT_ROOT, "models", "eval_plots")
os.makedirs(EVAL_DIR, exist_ok=True)

# ── Evaluate baseline ─────────────────────────────────────────────────
print("Evaluating baseline...")
baseline_metrics = evaluate_model(
    model_type="baseline",
    model_path=BASELINE_PATH,
    manifest_csv=MANIFEST_CSV,
    feature_dir=FEATURE_DIR,
    output_dir=EVAL_DIR,
)

# ── Evaluate CNN ──────────────────────────────────────────────────────
print("\nEvaluating CNN...")
cnn_metrics = evaluate_model(
    model_type="cnn",
    model_path=CNN_PATH,
    manifest_csv=MANIFEST_CSV,
    spec_dir=SPEC_DIR,
    output_dir=EVAL_DIR,
    device=DEVICE,
)

In [ ]:
# ── Display test-set confusion matrices side by side ──────────────────
from PIL import Image

cm_bl = os.path.join(EVAL_DIR, "confusion_matrix_baseline.png")
cm_cnn = os.path.join(EVAL_DIR, "confusion_matrix_cnn.png")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, path, title in [
    (axes[0], cm_bl, "Baseline SVM — Test Set"),
    (axes[1], cm_cnn, "CNN (EfficientNet-B0) — Test Set"),
]:
    if os.path.exists(path):
        img = Image.open(path)
        ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
fig.suptitle("Test Set Confusion Matrices", fontsize=14)
fig.tight_layout()
plt.show()

---
## Step 8 — Training Set Evaluation (Overfitting Check)

Evaluate both models on the **training data** to check for overfitting.
If train and test metrics are close, the models generalise well.

In [ ]:
from sklearn.metrics import (confusion_matrix as cm_fn, accuracy_score, f1_score,
                             classification_report)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from src.baseline_model import BaselineClassifier
from src.features import extract_mfcc_dataset
from src.cnn_model import FoxCNN
from src.dataset import FoxSpectrogramDataset

# ── Baseline on training set ──────────────────────────────────────────
clf = BaselineClassifier.load(BASELINE_PATH)
X, y = extract_mfcc_dataset(MANIFEST_CSV, feature_dir=FEATURE_DIR)
X_train, _, y_train, _ = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
y_pred_bl = clf.pipeline.predict(X_train)

# ── CNN on training set ───────────────────────────────────────────────
dev = torch.device(DEVICE)
ckpt = torch.load(CNN_PATH, map_location=dev, weights_only=False)
model = FoxCNN(backbone=ckpt.get("backbone", "efficientnet_b0"),
               pretrained=False, num_classes=2)
model.load_state_dict(ckpt["model_state_dict"])
model.to(dev).eval()

train_ds = FoxSpectrogramDataset(MANIFEST_CSV, SPEC_DIR, split="train",
                                  img_size=(128, 128), augment=False)
loader = DataLoader(train_ds, batch_size=32, shuffle=False)
all_labels, all_preds = [], []
with torch.no_grad():
    for imgs, labs in loader:
        all_labels.extend(labs.tolist())
        all_preds.extend(model(imgs.to(dev)).argmax(dim=1).cpu().tolist())
y_true_cnn = np.array(all_labels)
y_pred_cnn = np.array(all_preds)

# ── Side-by-side confusion matrices ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, title, yt, yp, cmap in [
    (axes[0], "SVM Baseline — Training Set", y_train, y_pred_bl, "Blues"),
    (axes[1], "CNN (EfficientNet-B0) — Training Set", y_true_cnn, y_pred_cnn, "Oranges"),
]:
    cm = cm_fn(yt, yp, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=["nonfox", "fox"],
                yticklabels=["nonfox", "fox"],
                ax=ax, annot_kws={"size": 18})
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("Actual", fontsize=12)
    ax.set_title(title, fontsize=13)
    acc = accuracy_score(yt, yp)
    f1 = f1_score(yt, yp, average="macro")
    ax.text(0.5, -0.15, f"Accuracy: {acc:.4f}  |  F1: {f1:.4f}",
            transform=ax.transAxes, ha="center", fontsize=11)

fig.suptitle(f"Training Set Evaluation (n = {len(y_train):,})", fontsize=15, y=1.02)
fig.tight_layout()
plt.show()

# ── Print detailed reports ────────────────────────────────────────────
print("=" * 60)
print("  SVM BASELINE — TRAINING SET")
print("=" * 60)
print(classification_report(y_train, y_pred_bl, target_names=["nonfox", "fox"]))

print("=" * 60)
print("  CNN (EfficientNet-B0) — TRAINING SET")
print("=" * 60)
print(classification_report(y_true_cnn, y_pred_cnn, target_names=["nonfox", "fox"]))

---
## Step 9 — Model Comparison

Side-by-side comparison of key metrics across both models.

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────
comparison_df = compare_models(baseline_metrics, cnn_metrics)
comparison_df

In [ ]:
# ── Bar chart comparison ──────────────────────────────────────────────
metrics_to_plot = ["accuracy", "f1_macro", "precision_macro", "recall_macro", "roc_auc"]
bl_vals = [baseline_metrics[k] for k in metrics_to_plot]
cnn_vals = [cnn_metrics[k] for k in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, bl_vals, width, label="Baseline SVM", color="steelblue")
bars2 = ax.bar(x + width/2, cnn_vals, width, label="CNN (EfficientNet-B0)", color="coral")

ax.set_ylabel("Score")
ax.set_title("Test Set Performance Comparison")
ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", " ").title() for m in metrics_to_plot], rotation=15)
ax.legend()
ax.set_ylim(0.95, 1.005)
ax.grid(axis="y", alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.4f}", xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

fig.tight_layout()
plt.show()

In [ ]:
# ── Display PR and ROC curves side by side ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col, curve_type in enumerate(["pr_curve", "roc_curve"]):
    ax = axes[col]
    for model_label, color in [("baseline", "steelblue"), ("cnn", "coral")]:
        curve_path = os.path.join(EVAL_DIR, f"{curve_type}_{model_label}.png")
        if os.path.exists(curve_path):
            img = Image.open(curve_path)
    # Just show the individual plot images
    for model_label in ["baseline", "cnn"]:
        curve_path = os.path.join(EVAL_DIR, f"{curve_type}_{model_label}.png")
        if os.path.exists(curve_path):
            img = Image.open(curve_path)
            ax.imshow(img)
            break
    title = "Precision-Recall Curve" if "pr" in curve_type else "ROC Curve"
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
plt.show()

# ── Summary ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"  Baseline SVM:       Accuracy={baseline_metrics['accuracy']:.4f}  F1={baseline_metrics['f1_macro']:.4f}  ROC-AUC={baseline_metrics['roc_auc']:.4f}")
print(f"  CNN EfficientNet:   Accuracy={cnn_metrics['accuracy']:.4f}  F1={cnn_metrics['f1_macro']:.4f}  ROC-AUC={cnn_metrics['roc_auc']:.4f}")
print(f"\n  Winner: {'CNN' if cnn_metrics['f1_macro'] > baseline_metrics['f1_macro'] else 'Baseline'} (by F1)")

---
## Step 10 — Interactive Demo (Gradio)

Launch a Gradio web app to upload/record audio and get real-time predictions.

The demo will be available at **http://127.0.0.1:7860**.

> **Note:** Stop this cell to terminate the server.

In [ ]:
from src.demo import launch_demo

launch_demo(
    cnn_path=CNN_PATH,
    baseline_path=BASELINE_PATH,
    port=7860,
    share=False,
)

# Fox Audio Detection — Project Setup

## Step 1: Create Project Directory Structure & Conda Environment

This cell creates the full `fox_detection/` project tree and verifies it.

In [2]:
import os

# Project root
PROJECT_ROOT = os.path.join(os.getcwd(), "fox_detection")

# Define directory structure
directories = [
    "data/raw/fox",
    "data/raw/nonfox",
    "data/clips/fox",
    "data/clips/nonfox",
    "data/features",
    "data/spectrograms/fox",
    "data/spectrograms/nonfox",
    "models/baseline",
    "models/cnn",
    "notebooks",
    "src",
]

# Create directories
for d in directories:
    path = os.path.join(PROJECT_ROOT, d)
    os.makedirs(path, exist_ok=True)
    print(f"✓ {d}/")

print(f"\nProject root: {PROJECT_ROOT}")

✓ data/raw/fox/
✓ data/raw/nonfox/
✓ data/clips/fox/
✓ data/clips/nonfox/
✓ data/features/
✓ data/spectrograms/fox/
✓ data/spectrograms/nonfox/
✓ models/baseline/
✓ models/cnn/
✓ notebooks/
✓ src/

Project root: /Users/dyx/Documents/TECHIN513A/Final/fox_detection


In [3]:
# Verify the complete project structure
for root, dirs, files in os.walk(PROJECT_ROOT):
    # Skip hidden dirs
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(PROJECT_ROOT, '').count(os.sep)
    indent = ' ' * 2 * level
    basename = os.path.basename(root)
    print(f'{indent}{basename}/')
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        if not file.startswith('.'):
            print(f'{subindent}{file}')

fox_detection/
  README.md
  environment.yml
  eval_train.py
  plot_cm.py
  report.md
  requirements.txt
  run_pipeline.sh
  tests/
    test_audio_utils.py
    __pycache__/
      test_audio_utils.cpython-314-pytest-9.0.2.pyc
  models/
    baseline/
      cm_val_gradient_boosting.png
      cm_val_random_forest.png
      cm_val_svm.png
      confusion_matrix.png
      model.pkl
      eval/
        confusion_matrix_train.png
    cnn/
      best.pt
      training_curves.png
      eval/
        confusion_matrix_train.png
    eval_plots/
  data/
    manifest.csv
    clips/
      nonfox/
        XC1002967_0000_1f485c4d0f4d.wav
        XC1002967_0000_38359d058564.wav
        XC1002967_0001_99580d0a2bd3.wav
        XC1002967_0001_f54bd5376460.wav
        XC1002967_0002_23fe67d3687d.wav
        XC1002967_0002_8f3170a9f4d3.wav
        XC1002967_0003_1e7f38975de3.wav
        XC1002967_0003_bf367763d247.wav
        XC1002967_0004_28293a3eee44.wav
        XC1002967_0004_ac8627039a69.wav
        XC10

## Environment Configuration

The `environment.yml` file specifies all dependencies for the conda environment:

```yaml
name: fox_detection
channels: [pytorch, conda-forge, defaults]
dependencies:
  python=3.10, librosa, soundfile, numpy, pandas, scikit-learn,
  matplotlib, seaborn, pytorch, torchvision, torchaudio,
  pillow, tqdm, jupyter, gradio (via pip)
```

**To create the environment:**
```bash
cd fox_detection
conda env create -f environment.yml
conda activate fox_detection
```

In [4]:
# Display the environment.yml contents
env_file = os.path.join(PROJECT_ROOT, "environment.yml")
with open(env_file, "r") as f:
    print(f.read())

name: fox_detection
channels:
  - pytorch
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - librosa
  - soundfile
  - numpy
  - pandas
  - scikit-learn
  - matplotlib
  - seaborn
  - pytorch::pytorch
  - pytorch::torchvision
  - pytorch::torchaudio
  - pillow
  - tqdm
  - jupyter
  - pip
  - pip:
      - gradio



## Source Module Placeholders

All source modules have been created under `fox_detection/src/` with placeholder functions:

| Module | Purpose |
|---|---|
| `audio_utils.py` | Audio loading, resampling, normalization |
| `segmentation.py` | Splitting recordings into 3-second clips |
| `features.py` | MFCC extraction & spectrogram generation |
| `dataset.py` | PyTorch Dataset for spectrogram images |
| `baseline_model.py` | scikit-learn baseline training/inference |
| `cnn_model.py` | CNN architecture (PyTorch) |
| `train_cnn.py` | CNN training loop |
| `evaluate.py` | Metrics computation & visualization |
| `demo.py` | Gradio interactive demo |

Five placeholder notebooks are in `fox_detection/notebooks/` for each project stage.

In [5]:
# List all source files
src_dir = os.path.join(PROJECT_ROOT, "src")
print("Source modules:")
for f in sorted(os.listdir(src_dir)):
    if f.endswith('.py'):
        print(f"  📄 {f}")

# List all notebooks
nb_dir = os.path.join(PROJECT_ROOT, "notebooks")
print("\nNotebooks:")
for f in sorted(os.listdir(nb_dir)):
    if f.endswith('.ipynb'):
        print(f"  📓 {f}")

print("\n✅ Project setup complete!")

Source modules:
  📄 __init__.py
  📄 audio_utils.py
  📄 baseline_model.py
  📄 cnn_model.py
  📄 dataset.py
  📄 demo.py
  📄 download_data.py
  📄 evaluate.py
  📄 features.py
  📄 segmentation.py
  📄 train_cnn.py

Notebooks:
  📓 01_data_exploration.ipynb
  📓 02_preprocessing.ipynb
  📓 03_baseline_model.ipynb
  📓 04_cnn_model.ipynb
  📓 05_evaluation.ipynb

✅ Project setup complete!


---

## Step 2: `src/audio_utils.py`

Core audio utilities — loading, normalisation, silence trimming, log-mel spectrograms, spectrogram image export, and MFCC feature extraction.

In [6]:
# Run the audio_utils self-test via its __main__ block
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "src.audio_utils"],
    capture_output=True, text=True,
    cwd=os.path.join(os.getcwd(), "fox_detection") if "fox_detection" not in os.getcwd() else os.getcwd(),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

=== audio_utils self-test ===

✓ normalise_audio  – peak=1.0000
✓ trim_silence     – 110250 → 68096 samples
✓ log_mel_spec     – shape (128, 130)
✓ save_spec_image  – /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/tmpicpkox_r.png (2555 bytes)
✓ mfcc_features    – shape (80,)

=== all tests passed ===



In [7]:
# Interactive test — import functions and exercise them on a synthetic sine wave
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "fox_detection"))

import numpy as np
from src.audio_utils import (
    normalise_audio,
    trim_silence,
    compute_log_mel_spectrogram,
    save_spectrogram_image,
    compute_mfcc_features,
)

SR = 22050
DURATION = 3.0
t = np.linspace(0, DURATION, int(SR * DURATION), endpoint=False)
sine = (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)

# 1. Normalise
normed = normalise_audio(sine)
print(f"normalise_audio  → peak = {np.max(np.abs(normed)):.4f}")

# 2. Trim silence (add 1 s padding of zeros)
padded = np.concatenate([np.zeros(SR, dtype=np.float32), sine, np.zeros(SR, dtype=np.float32)])
trimmed = trim_silence(padded, SR)
print(f"trim_silence     → {len(padded):,} → {len(trimmed):,} samples")

# 3. Log-mel spectrogram
spec = compute_log_mel_spectrogram(sine, SR)
print(f"log_mel_spec     → shape {spec.shape}, range [{spec.min():.1f}, {spec.max():.1f}] dB")

# 4. Save spectrogram image
import tempfile
tmp_path = os.path.join(tempfile.gettempdir(), "test_spec.png")
save_spectrogram_image(spec, tmp_path)
print(f"save_spec_image  → {tmp_path} ({os.path.getsize(tmp_path)} bytes)")

# 5. MFCC features
feat = compute_mfcc_features(sine, SR, n_mfcc=40)
print(f"mfcc_features    → shape {feat.shape}  (first 5: {feat[:5]})")

print("\n✅ All audio_utils functions work correctly!")

normalise_audio  → peak = 1.0000
trim_silence     → 110,250 → 68,096 samples
log_mel_spec     → shape (128, 130), range [-80.0, 0.0] dB
save_spec_image  → /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/test_spec.png (2555 bytes)
mfcc_features    → shape (80,)  (first 5: [-475.57126     42.825       28.526556    14.072545    -2.1203027])

✅ All audio_utils functions work correctly!


---

## Step 3: `src/segmentation.py`

Audio segmentation with two strategies and a directory-level pipeline:

| Function | Description |
|---|---|
| `segment_fixed(y, sr, clip_duration, overlap)` | Fixed-length overlapping clips, zero-padded tail |
| `segment_energy(y, sr, …)` | RMS-energy event detection, merging, padding |
| `process_directory(input_dir, output_dir, label, …)` | Load → normalise → segment → save WAVs + DataFrame rows |
| `build_manifest(fox_dir, nonfox_dir, …)` | Process both classes, write combined `manifest.csv` |

**CLI usage:**
```bash
python -m src.segmentation --fox_dir data/raw/fox --nonfox_dir data/raw/nonfox \
                           --out_dir data/clips --manifest data/manifest.csv
```

In [8]:
# Test segmentation functions on a synthetic waveform
from src.segmentation import segment_fixed, segment_energy

# Reuse the sine wave from Step 2: 3 s @ 22050 Hz
print("── segment_fixed ──")
clips_fixed = segment_fixed(sine, SR, clip_duration=1.0, overlap=0.5)
for i, c in enumerate(clips_fixed):
    print(f"  clip {i}: {len(c)} samples ({len(c)/SR:.2f} s)")
print(f"  Total fixed clips: {len(clips_fixed)}")

# Build a test signal for energy-based segmentation:
# 0.5s silence | 1s tone | 0.5s silence | 0.5s tone | 0.5s silence
tone_1 = 0.8 * np.sin(2 * np.pi * 440 * np.arange(int(SR * 1.0)) / SR).astype(np.float32)
tone_2 = 0.8 * np.sin(2 * np.pi * 880 * np.arange(int(SR * 0.5)) / SR).astype(np.float32)
silence = np.zeros(int(SR * 0.5), dtype=np.float32)

test_signal = np.concatenate([silence, tone_1, silence, tone_2, silence])
print(f"\n── segment_energy ──")
print(f"  Test signal: {len(test_signal)} samples ({len(test_signal)/SR:.2f} s)")

clips_energy = segment_energy(test_signal, SR, energy_threshold_db=-30, min_event_duration=0.3)
for i, c in enumerate(clips_energy):
    print(f"  event {i}: {len(c)} samples ({len(c)/SR:.2f} s)")
print(f"  Total energy clips: {len(clips_energy)}")

print("\n✅ Segmentation functions verified!")

── segment_fixed ──
  clip 0: 22050 samples (1.00 s)
  clip 1: 22050 samples (1.00 s)
  clip 2: 22050 samples (1.00 s)
  clip 3: 22050 samples (1.00 s)
  clip 4: 22050 samples (1.00 s)
  clip 5: 22050 samples (1.00 s)
  Total fixed clips: 6

── segment_energy ──
  Test signal: 66150 samples (3.00 s)
  event 0: 28474 samples (1.29 s)
  event 1: 17210 samples (0.78 s)
  Total energy clips: 2

✅ Segmentation functions verified!


---

## Step 4: `src/features.py`

Batch extraction of MFCC features and log-mel spectrogram images from a manifest CSV.

| Function | Description |
|---|---|
| `extract_mfcc_dataset(manifest_csv, …)` | Load each clip → MFCC feature vector → cache as `.npy` → return `(X, y)` arrays |
| `extract_spectrogram_dataset(manifest_csv, …)` | Load each clip → log-mel spectrogram → save as grayscale PNG under `<spec_dir>/<label>/` |

Both functions skip already-processed files for fast re-runs.

**CLI usage:**
```bash
python -m src.features --manifest data/manifest.csv \
                       --feature_dir data/features/ \
                       --spec_dir data/spectrograms/ \
                       --mode both
```

In [9]:
# Smoke-test features.py using a tiny synthetic manifest
# 1. Create two short WAV clips (fox + nonfox) in a temp directory
import tempfile, shutil, pandas as pd, soundfile as sf
from pathlib import Path

tmp_root = tempfile.mkdtemp(prefix="fox_feat_test_")
clips_dir = os.path.join(tmp_root, "clips")
os.makedirs(os.path.join(clips_dir, "fox"), exist_ok=True)
os.makedirs(os.path.join(clips_dir, "nonfox"), exist_ok=True)

# Synthetic clips: 3 s sine waves at different frequencies
for label, freq, idx in [("fox", 440, 0), ("fox", 500, 1),
                          ("nonfox", 220, 0), ("nonfox", 300, 1)]:
    t_arr = np.linspace(0, 3.0, int(SR * 3), endpoint=False)
    clip = (0.5 * np.sin(2 * np.pi * freq * t_arr)).astype(np.float32)
    sf.write(os.path.join(clips_dir, label, f"clip_{idx:04d}.wav"), clip, SR)

# 2. Build a manifest
rows = []
for label in ("fox", "nonfox"):
    for f in sorted(os.listdir(os.path.join(clips_dir, label))):
        rows.append({
            "file_id": Path(f).stem,
            "source_file": f,
            "clip_path": os.path.join("clips", label, f),
            "label": label,
            "start_sec": 0.0,
            "end_sec": 3.0,
        })
manifest_csv = os.path.join(tmp_root, "manifest.csv")
pd.DataFrame(rows).to_csv(manifest_csv, index=False)
print(f"Test manifest ({len(rows)} rows):")
print(pd.read_csv(manifest_csv).to_string(index=False))

# 3. Run MFCC extraction
from src.features import extract_mfcc_dataset, extract_spectrogram_dataset

feat_dir = os.path.join(tmp_root, "features")
X, y = extract_mfcc_dataset(manifest_csv, feature_dir=feat_dir, sr=SR)
print(f"\nMFCC: X.shape={X.shape}, y={y}")

# 4. Run spectrogram extraction
spec_dir = os.path.join(tmp_root, "spectrograms")
extract_spectrogram_dataset(manifest_csv, spec_dir=spec_dir, sr=SR)
for label in ("fox", "nonfox"):
    pngs = os.listdir(os.path.join(spec_dir, label))
    print(f"  {label}/: {len(pngs)} PNGs → {pngs}")

# 5. Re-run to verify caching (should skip all)
print("\n── Re-run (should hit cache) ──")
X2, y2 = extract_mfcc_dataset(manifest_csv, feature_dir=feat_dir, sr=SR)
extract_spectrogram_dataset(manifest_csv, spec_dir=spec_dir, sr=SR)

# Clean up
shutil.rmtree(tmp_root)
print("\n✅ features.py verified!")

Test manifest (4 rows):
  file_id   source_file                  clip_path  label  start_sec  end_sec
clip_0000 clip_0000.wav    clips/fox/clip_0000.wav    fox        0.0      3.0
clip_0001 clip_0001.wav    clips/fox/clip_0001.wav    fox        0.0      3.0
clip_0000 clip_0000.wav clips/nonfox/clip_0000.wav nonfox        0.0      3.0
clip_0001 clip_0001.wav clips/nonfox/clip_0001.wav nonfox        0.0      3.0


MFCC features: 100%|██████████| 4/4 [00:15<00:00,  3.76s/it]


MFCC dataset: X (4, 80), y (4,)  (fox=2, nonfox=2)

MFCC: X.shape=(4, 80), y=[1 1 0 0]


Spectrograms: 100%|██████████| 4/4 [00:00<00:00, 51.57it/s]


Spectrograms: 4 created, 0 skipped (cached)
  fox/: 2 PNGs → ['clip_0001.png', 'clip_0000.png']
  nonfox/: 2 PNGs → ['clip_0001.png', 'clip_0000.png']

── Re-run (should hit cache) ──


MFCC features: 100%|██████████| 4/4 [00:00<00:00, 2331.14it/s]


MFCC dataset: X (4, 80), y (4,)  (fox=2, nonfox=2)


Spectrograms: 100%|██████████| 4/4 [00:00<00:00, 2717.40it/s]

Spectrograms: 0 created, 4 skipped (cached)

✅ features.py verified!


---

## Step 5: `src/baseline_model.py`

Scikit-learn baseline classifiers with a `BaselineClassifier` wrapper class.

| Component | Description |
|---|---|
| `BaselineClassifier(model_type)` | Wraps `StandardScaler` + classifier in a sklearn `Pipeline`. Supports `'svm'`, `'random_forest'`, `'gradient_boosting'` |
| `.train(X, y)` | Fit pipeline, print training accuracy |
| `.evaluate(X, y)` | Return `{accuracy, precision, recall, f1, classification_report}`, save confusion-matrix heatmap |
| `.save(path)` / `.load(path)` | Pickle round-trip |
| `train_baseline(manifest, …)` | Load MFCCs → 70/15/15 split → train all 3 models → pick best by val F1 → evaluate on test → save |

**CLI:**
```bash
python -m src.baseline_model --manifest data/manifest.csv \
                             --feature_dir data/features/ \
                             --model_dir models/baseline/
```

In [10]:
# Smoke-test BaselineClassifier on synthetic separable data
from src.baseline_model import BaselineClassifier
import tempfile, shutil

rng = np.random.RandomState(42)
n_per_class = 100
X_fox    = rng.randn(n_per_class, 80).astype(np.float32) + 2.0
X_nonfox = rng.randn(n_per_class, 80).astype(np.float32) - 2.0
X_syn = np.vstack([X_fox, X_nonfox])
y_syn = np.array([1]*n_per_class + [0]*n_per_class, dtype=np.int64)

# Shuffle
idx = rng.permutation(len(y_syn))
X_syn, y_syn = X_syn[idx], y_syn[idx]

# Split 70/30
split = int(0.7 * len(y_syn))
X_tr, X_te = X_syn[:split], X_syn[split:]
y_tr, y_te = y_syn[:split], y_syn[split:]

tmp_dir = tempfile.mkdtemp(prefix="bl_nb_test_")

for mt in ("svm", "random_forest", "gradient_boosting"):
    print(f"\n── {mt} ──")
    clf = BaselineClassifier(model_type=mt)
    clf.train(X_tr, y_tr)
    m = clf.evaluate(X_te, y_te,
                     save_cm_path=os.path.join(tmp_dir, f"cm_{mt}.png"))
    print(f"  Test → acc={m['accuracy']:.3f}  prec={m['precision']:.3f}  "
          f"rec={m['recall']:.3f}  f1={m['f1']:.3f}")

    # Save/load round-trip
    pkl = os.path.join(tmp_dir, f"{mt}.pkl")
    clf.save(pkl)
    clf2 = BaselineClassifier.load(pkl)
    assert np.array_equal(clf2.pipeline.predict(X_te), clf.pipeline.predict(X_te))
    print("  ✓ save/load round-trip OK")

shutil.rmtree(tmp_dir)
print("\n✅ baseline_model.py verified!")


── svm ──
[svm] Training accuracy: 1.0000
  Confusion matrix saved to /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/bl_nb_test_54887b4c/cm_svm.png
  Test → acc=1.000  prec=1.000  rec=1.000  f1=1.000
  Model saved to /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/bl_nb_test_54887b4c/svm.pkl
  ✓ save/load round-trip OK

── random_forest ──
[random_forest] Training accuracy: 1.0000
  Confusion matrix saved to /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/bl_nb_test_54887b4c/cm_random_forest.png
  Test → acc=1.000  prec=1.000  rec=1.000  f1=1.000
  Model saved to /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/bl_nb_test_54887b4c/random_forest.pkl
  ✓ save/load round-trip OK

── gradient_boosting ──
[gradient_boosting] Training accuracy: 1.0000
  Confusion matrix saved to /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/bl_nb_test_54887b4c/cm_gradient_boosting.png
  Test → acc=1.000  prec=1.000  rec=1.000  f1=1.000
  Model saved to /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T

---

## Step 6: `src/dataset.py`

PyTorch `Dataset` for spectrogram images with stratified splitting and augmentation.

| Component | Description |
|---|---|
| `FoxSpectrogramDataset` | Loads spectrogram PNGs, stratified 70/15/15 split, outputs `(3, H, W)` tensors |
| Augmentation (train) | Random horizontal flip, frequency masking (≤20 rows), time masking (≤20 cols), Gaussian noise (σ=0.01) |
| Normalisation | `[0,1]` → ImageNet mean/std (`[0.485,0.456,0.406]` / `[0.229,0.224,0.225]`) |
| `get_class_weights()` | Inverse-frequency weights for `nn.CrossEntropyLoss(weight=…)` |

In [11]:
# Smoke-test FoxSpectrogramDataset with synthetic PNGs
import tempfile, shutil
import pandas as pd
from PIL import Image

tmp_ds = tempfile.mkdtemp(prefix="ds_nb_")
spec_dir_test = os.path.join(tmp_ds, "specs")
for lb in ("fox", "nonfox"):
    os.makedirs(os.path.join(spec_dir_test, lb))

rows = []
for lb, bv in [("fox", 200), ("nonfox", 50)]:
    for i in range(20):
        fid = f"{lb}_{i:04d}"
        arr = np.full((64, 64), bv + i, dtype=np.uint8)
        Image.fromarray(arr, mode="L").save(os.path.join(spec_dir_test, lb, f"{fid}.png"))
        rows.append({"file_id": fid, "source_file": f"{fid}.wav",
                      "clip_path": f"clips/{lb}/{fid}.wav", "label": lb,
                      "start_sec": 0.0, "end_sec": 3.0})
csv_test = os.path.join(tmp_ds, "manifest.csv")
pd.DataFrame(rows).to_csv(csv_test, index=False)

from src.dataset import FoxSpectrogramDataset

for split in ("train", "val", "test"):
    ds = FoxSpectrogramDataset(csv_test, spec_dir_test, split=split,
                               img_size=(64, 64), augment=(split == "train"))
    img, label = ds[0]
    print(f"{split:5s}: len={len(ds):3d}  shape={tuple(img.shape)}  "
          f"label={label}  range=[{img.min():.2f}, {img.max():.2f}]")

# Class weights
cw = FoxSpectrogramDataset.get_class_weights(
    FoxSpectrogramDataset(csv_test, spec_dir_test, split="train", img_size=(64,64)).labels
)
print(f"\nClass weights: {cw}")

shutil.rmtree(tmp_ds)
print("\n✅ dataset.py verified!")

train: len= 28  shape=(3, 64, 64)  label=1  range=[-2.27, 1.94]
val  : len=  6  shape=(3, 64, 64)  label=1  range=[1.31, 1.68]
test : len=  6  shape=(3, 64, 64)  label=0  range=[-1.19, -0.86]

Class weights: tensor([1., 1.])

✅ dataset.py verified!


---

## Step 7: `src/cnn_model.py` & `src/train_cnn.py`

### CNN Architecture (`FoxCNN`)

| Backbone | Source | Final head |
|---|---|---|
| `efficientnet_b0` | `torchvision.models` | `Dropout(p) → Linear(1280, 2)` |
| `resnet18` | `torchvision.models` | `Dropout(p) → Linear(512, 2)` |
| `mobilenet_v3_small` | `torchvision.models` | `Dropout(p) → Linear(1024, 2)` |

### Training Loop (`train_cnn`)

- **Data**: `FoxSpectrogramDataset` with augmentation on train split
- **Loss**: Weighted `CrossEntropyLoss` (inverse-frequency class weights)
- **Optimiser**: `AdamW` (weight_decay=1e-4)
- **Scheduler**: `CosineAnnealingLR`
- **Early stopping**: patience=7 epochs by val F1
- **Outputs**: `best.pt` checkpoint + `training_curves.png`

**CLI:**
```bash
python -m src.train_cnn --manifest data/manifest.csv --spec_dir data/spectrograms/ \
                        --backbone efficientnet_b0 --epochs 30 --batch_size 32
```

In [12]:
# Test FoxCNN forward pass for all three backbones
import torch
from src.cnn_model import FoxCNN

dummy_input = torch.randn(2, 3, 128, 128)

for bb in ("efficientnet_b0", "resnet18", "mobilenet_v3_small"):
    model = FoxCNN(backbone=bb, pretrained=False, num_classes=2, dropout=0.3)
    out = model(dummy_input)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"✓ {bb:25s} → output {tuple(out.shape)}  params={n_params:,}")

print("\n✅ FoxCNN verified for all backbones!")

✓ efficientnet_b0           → output (2, 2)  params=4,010,110
✓ resnet18                  → output (2, 2)  params=11,177,538
✓ mobilenet_v3_small        → output (2, 2)  params=1,519,906

✅ FoxCNN verified for all backbones!


In [13]:
# Quick 2-epoch training test on synthetic spectrograms
import tempfile, shutil
from PIL import Image
from src.train_cnn import train_cnn

tmp_cnn = tempfile.mkdtemp(prefix="cnn_nb_")
spec_dir_cnn = os.path.join(tmp_cnn, "specs")
for lb in ("fox", "nonfox"):
    os.makedirs(os.path.join(spec_dir_cnn, lb))

rows = []
for lb, bv in [("fox", 200), ("nonfox", 50)]:
    for i in range(16):
        fid = f"{lb}_{i:04d}"
        arr = np.random.randint(bv, bv + 30, (64, 64), dtype=np.uint8)
        Image.fromarray(arr, mode="L").save(os.path.join(spec_dir_cnn, lb, f"{fid}.png"))
        rows.append({"file_id": fid, "source_file": f"{fid}.wav",
                      "clip_path": f"clips/{lb}/{fid}.wav", "label": lb,
                      "start_sec": 0.0, "end_sec": 3.0})
csv_cnn = os.path.join(tmp_cnn, "manifest.csv")
pd.DataFrame(rows).to_csv(csv_cnn, index=False)

best = train_cnn(
    manifest_csv=csv_cnn, spec_dir=spec_dir_cnn,
    model_dir=os.path.join(tmp_cnn, "models"),
    backbone="resnet18", pretrained=False,
    epochs=2, batch_size=8, img_size=(64, 64), device="cpu",
)
ckpt = torch.load(best, map_location="cpu", weights_only=False)
print(f"\nCheckpoint: epoch={ckpt['epoch']}  val_f1={ckpt['val_f1']:.4f}")

shutil.rmtree(tmp_cnn)
print("✅ train_cnn verified!")

Device: cpu
Train samples: 22  |  Val samples: 5
Class weights: [1.0, 1.0]


Epoch   1/2  train_loss=0.2930  val_loss=1.4314  val_f1=0.7500  lr=0.000500  (0.3s)
  ★ Best model saved (val_f1=0.7500)


Epoch   2/2  train_loss=0.0326  val_loss=0.0085  val_f1=1.0000  lr=0.000000  (0.2s)
  ★ Best model saved (val_f1=1.0000)

Training complete. Best epoch: 2  Best val F1: 1.0000
Training curves saved to /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/cnn_nb_u964cucm/models/training_curves.png

Checkpoint: epoch=2  val_f1=1.0000
✅ train_cnn verified!


## Step 8 — `src/evaluate.py`

Unified evaluation module that works for **both** baseline (scikit-learn) and CNN (PyTorch) models.

### Public API

| Function | Purpose |
|---|---|
| `evaluate_model(model_type, model_path, manifest_csv, ...)` | Run inference on the **test** split, compute 7 metrics, save 3 diagnostic plots |
| `compare_models(baseline_metrics, cnn_metrics)` | Print a side-by-side pandas DataFrame comparison table |

### Metrics computed
`accuracy`, `precision_macro`, `recall_macro`, `f1_macro`, `f1_weighted`, `pr_auc`, `roc_auc`

### Plots saved
- **Confusion matrix** heatmap (`.png`)
- **Precision-Recall curve** with AP (`.png`)
- **ROC curve** with AUC (`.png`)

### CLI
```bash
python -m src.evaluate --model_type baseline --model_path models/baseline_best.pkl \
       --manifest data/manifest.csv --feature_dir data/features
python -m src.evaluate --model_type cnn --model_path models/cnn/best.pt \
       --manifest data/manifest.csv --spec_dir data/spectrograms
```

In [14]:
# ── Step 8: Smoke-test src/evaluate.py ─────────────────────────────────
import os, sys, shutil, tempfile
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(PROJECT_ROOT, "fox_detection"))

# ── Build a tiny synthetic environment ───────────────────────────────
tmp = tempfile.mkdtemp(prefix="eval_test_")
print(f"[tmp dir] {tmp}")

rows = []
for i in range(10):
    rows.append({"clip_path": f"clips/fox_{i}.wav", "label": "fox"})
    rows.append({"clip_path": f"clips/nonfox_{i}.wav", "label": "nonfox"})
manifest_csv = os.path.join(tmp, "manifest.csv")
pd.DataFrame(rows).to_csv(manifest_csv, index=False)

# ── Monkey-patch inference to inject known outputs ───────────────────
import src.evaluate as ev

def _fake_baseline(*a, **kw):
    y_true = np.array([1, 0, 1, 0, 1, 0])
    y_pred = np.array([1, 0, 1, 1, 1, 0])
    y_prob = np.array([0.9, 0.1, 0.85, 0.6, 0.95, 0.2])
    return y_true, y_pred, y_prob

def _fake_cnn(*a, **kw):
    y_true = np.array([1, 0, 1, 0, 1, 0])
    y_pred = np.array([1, 0, 1, 0, 0, 0])
    y_prob = np.array([0.95, 0.05, 0.9, 0.1, 0.4, 0.15])
    return y_true, y_pred, y_prob

ev._infer_baseline = _fake_baseline
ev._infer_cnn      = _fake_cnn

out_dir = os.path.join(tmp, "eval_output")

# ── Evaluate both model types ────────────────────────────────────────
base_m = ev.evaluate_model("baseline", "dummy.pkl", manifest_csv,
                           feature_dir="x", output_dir=out_dir)
cnn_m  = ev.evaluate_model("cnn", "dummy.pt", manifest_csv,
                           spec_dir="x", output_dir=out_dir)

# ── Side-by-side comparison ──────────────────────────────────────────
df = ev.compare_models(base_m, cnn_m)

# ── Verify generated plot files ──────────────────────────────────────
for fn in ["confusion_matrix_baseline.png", "pr_curve_baseline.png",
           "roc_curve_baseline.png", "confusion_matrix_cnn.png",
           "pr_curve_cnn.png", "roc_curve_cnn.png"]:
    fp = os.path.join(out_dir, fn)
    assert os.path.isfile(fp), f"Missing: {fn}"
    print(f"  ✓ {fn}  ({os.path.getsize(fp):,} bytes)")

shutil.rmtree(tmp)
print("\n✅ evaluate.py smoke-test passed!")

[tmp dir] /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/eval_test_6d6rbpy4

══════════════════════════════════════════════════
  Evaluation — BASELINE
══════════════════════════════════════════════════
  accuracy            : 0.8333
  precision_macro     : 0.8750
  recall_macro        : 0.8333
  f1_macro            : 0.8286
  f1_weighted         : 0.8286
  pr_auc              : 1.0000
  roc_auc             : 1.0000

              precision    recall  f1-score   support

      nonfox       1.00      0.67      0.80         3
         fox       0.75      1.00      0.86         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6

  Saved confusion_matrix_baseline.png
  Saved pr_curve_baseline.png
  Saved roc_curve_baseline.png

══════════════════════════════════════════════════
  Evaluation — CNN
══════════════════════════════════════════════════
  accuracy            : 0.833

## Step 9 — `src/demo.py` (Gradio Web App)

A Gradio-powered web interface for real-time fox vocalisation detection.

### Layout
| Widget | Description |
|---|---|
| **Audio input** | Upload `.wav` `.mp3` `.ogg` `.flac` or record via microphone |
| **Model selector** | Dropdown: *CNN (EfficientNet-B0)* / *Baseline SVM* |
| **Analyse button** | Triggers the full inference pipeline |

### Pipeline (on submit)
1. **Load & normalise** audio via `audio_utils.load_audio` + `normalise_audio`.
2. **Segment** into 3-second clips (`segment_fixed`, overlap=0.5 s).
3. **Per-clip inference** with the selected model (CNN → spectrogram → FoxCNN, or Baseline → MFCC → SVM).
4. **Aggregate**: label file as 🦊 FOX if > 30 % of clips are fox.
5. **Outputs**: prediction label + confidence + waveform with colour-coded clips + log-mel spectrogram of the most-confident fox clip.

### CLI
```bash
python src/demo.py --cnn_model models/cnn/best.pt \
                   --baseline_model models/baseline/model.pkl \
                   --manifest data/manifest.csv --port 7860
```

In [15]:
# ── Step 9: Smoke-test src/demo.py (no server launch) ─────────────────
import os, sys, tempfile, shutil
import numpy as np
import soundfile as sf

sys.path.insert(0, os.path.join(PROJECT_ROOT, "fox_detection"))

# 1. Synthesise a 6-second test WAV
tmp = tempfile.mkdtemp(prefix="demo_test_")
t = np.linspace(0, 6.0, int(SR * 6.0), endpoint=False)
signal = (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
wav_path = os.path.join(tmp, "test.wav")
sf.write(wav_path, signal, SR)
print(f"[1] Test WAV: {wav_path}")

# 2. Fake baseline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import pickle

rng = np.random.RandomState(42)
X_tr = rng.randn(40, 80).astype(np.float32)
y_tr = np.array([0]*20 + [1]*20)
pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=42))])
pipe.fit(X_tr, y_tr)
pkl_path = os.path.join(tmp, "baseline.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump({"model_type": "svm", "pipeline": pipe}, f)
print(f"[2] Fake baseline: {pkl_path}")

# 3. Fake CNN checkpoint
import torch
from src.cnn_model import FoxCNN
cnn = FoxCNN(backbone="efficientnet_b0", pretrained=False, num_classes=2)
pt_path = os.path.join(tmp, "best.pt")
torch.save({"backbone": "efficientnet_b0", "model_state_dict": cnn.state_dict()}, pt_path)
print(f"[3] Fake CNN: {pt_path}")

# 4. Load models & run predictions
from src.demo import load_models, predict, _MODEL_CACHE
load_models(cnn_path=pt_path, baseline_path=pkl_path)

for model_name in ["Baseline SVM", "CNN (EfficientNet-B0)"]:
    label, conf, wf_fig, sp_fig = predict(wav_path, model_name)
    print(f"\n--- {model_name} ---")
    print(f"  Prediction: {label}  |  Confidence: {conf}")
    import matplotlib.pyplot as plt
    plt.close(wf_fig); plt.close(sp_fig)
    print(f"  ✓ OK")

shutil.rmtree(tmp)
print("\n✅ demo.py smoke-test passed!")

[1] Test WAV: /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/demo_test_jbenk1n5/test.wav
[2] Fake baseline: /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/demo_test_jbenk1n5/baseline.pkl


[3] Fake CNN: /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/demo_test_jbenk1n5/best.pt
[demo] CNN loaded from /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/demo_test_jbenk1n5/best.pt  (device=mps)
[demo] Baseline loaded from /var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/demo_test_jbenk1n5/baseline.pkl

--- Baseline SVM ---
  Prediction: No fox detected  |  Confidence: 37.5%
  ✓ OK

--- CNN (EfficientNet-B0) ---
  Prediction: No fox detected  |  Confidence: 50.5%
  ✓ OK

✅ demo.py smoke-test passed!


## Step 10 – Analysis Notebooks

Five self-contained Jupyter notebooks live in `fox_detection/notebooks/`:

| # | Notebook | Purpose |
|---|----------|---------|
| 1 | `01_data_exploration.ipynb` | Load manifest, class-distribution bar chart, waveform + spectrogram plots, duration histogram, average spectrogram per class |
| 2 | `02_preprocessing.ipynb` | Demonstrate `load_audio → normalise → trim_silence` pipeline, before/after spectrograms, `segment_fixed` with all clips displayed |
| 3 | `03_baseline_model.ipynb` | Extract/load MFCCs, 70/15/15 split, train SVM / Random-Forest / Gradient-Boosting, confusion matrices, F1 bar chart |
| 4 | `04_cnn_model.ipynb` | Display augmented images, train CNN with `train_cnn()`, learning-curve plots, per-class metrics from saved checkpoint |
| 5 | `05_evaluation.ipynb` | Evaluate both model families, `compare_models()` summary table, combined PR + ROC curves on one figure |

All notebooks include **try / except** guards so they fall back to synthetic data when real audio or model artefacts are missing.

In [16]:
# ── Step 10: Verify notebooks exist and are valid ──────────────────────
import json, pathlib

nb_dir = pathlib.Path(PROJECT_ROOT) / "notebooks"
expected = [
    "01_data_exploration.ipynb",
    "02_preprocessing.ipynb",
    "03_baseline_model.ipynb",
    "04_cnn_model.ipynb",
    "05_evaluation.ipynb",
]

for name in expected:
    p = nb_dir / name
    assert p.exists(), f"Missing: {p}"
    nb = json.loads(p.read_text())
    n_cells = len(nb["cells"])
    print(f"✓ {name:35s}  {n_cells:>3} cells  nbformat {nb['nbformat']}")

print("\n✅ All 5 analysis notebooks present and valid.")

✓ 01_data_exploration.ipynb             11 cells  nbformat 4
✓ 02_preprocessing.ipynb                10 cells  nbformat 4
✓ 03_baseline_model.ipynb               12 cells  nbformat 4
✓ 04_cnn_model.ipynb                    12 cells  nbformat 4
✓ 05_evaluation.ipynb                   12 cells  nbformat 4

✅ All 5 analysis notebooks present and valid.


## Step 11 – Pipeline Script & Unit Tests

### `run_pipeline.sh`
A single `bash` script that executes the full pipeline end-to-end (segment → extract features → train baseline → train CNN → evaluate both → launch demo). Run from the `fox_detection/` directory:
```bash
cd fox_detection && bash run_pipeline.sh
```

### `tests/test_audio_utils.py`
17 pytest tests covering four core components:

| Class | What is tested |
|-------|----------------|
| `TestLoadAudio` | correct SR, float32 dtype, mono 1-D, sample count, resampling |
| `TestNormaliseAudio` | output ∈ [-1, 1], peak == 1, silent signal, dtype preservation |
| `TestComputeMfccFeatures` | shape (80,), custom n_mfcc, float32, no NaNs |
| `TestSegmentFixed` | exact clip length, overlap handling, clip count, zero-padding |

In [17]:
# ── Step 11: Verify run_pipeline.sh & run pytest ───────────────────────
import subprocess, pathlib

project = pathlib.Path(PROJECT_ROOT)

# 11a – run_pipeline.sh exists and is executable
sh = project / "run_pipeline.sh"
assert sh.exists(), f"Missing: {sh}"
print(f"✓ {sh.name} exists ({sh.stat().st_size} bytes)")

# 11b – tests/test_audio_utils.py exists
test_file = project / "tests" / "test_audio_utils.py"
assert test_file.exists(), f"Missing: {test_file}"
print(f"✓ {test_file.relative_to(project)} exists")

# 11c – run pytest
result = subprocess.run(
    ["python", "-m", "pytest", str(test_file), "-v", "--tb=short"],
    capture_output=True, text=True, cwd=str(project),
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0, "pytest failed!"
print("✅ All 17 tests passed.")

✓ run_pipeline.sh exists (2091 bytes)
✓ tests/test_audio_utils.py exists
============================= test session starts ==============================
platform darwin -- Python 3.14.1, pytest-9.0.2, pluggy-1.6.0 -- /Users/dyx/Documents/TECHIN513A/Final/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/dyx/Documents/TECHIN513A/Final/fox_detection
plugins: anyio-4.12.1
collecting ... collected 17 items

tests/test_audio_utils.py::TestLoadAudio::test_returns_correct_sr PASSED [  5%]
tests/test_audio_utils.py::TestLoadAudio::test_returns_float32 PASSED    [ 11%]
tests/test_audio_utils.py::TestLoadAudio::test_mono_1d PASSED            [ 17%]
tests/test_audio_utils.py::TestLoadAudio::test_sample_count_roughly_correct PASSED [ 23%]
tests/test_audio_utils.py::TestLoadAudio::test_resample PASSED           [ 29%]
tests/test_audio_utils.py::TestNormaliseAudio::test_output_in_range PASSED [ 35%]
tests/test_audio_utils.py::TestNormaliseAudio::test_peak_is_one PASSED   [ 41%]
tests/test_au

## Step 12 – Data Download (`src/download_data.py`)

Downloads fox and non-fox audio from the **Xeno-canto API v3**.

| Query | Target | Destination |
|-------|--------|-------------|
| `gen:vulpes+sp:vulpes+q:A` | Fox (quality A) | `data/raw/fox/XC{id}.mp3` |
| `gen:vulpes+sp:vulpes+q:B` | Fox (quality B) | `data/raw/fox/XC{id}.mp3` |
| `grp:birds+cnt:europe+q:A` | European birds (negative) | `data/raw/nonfox/XC{id}.mp3` |
| `grp:"land mammals"+q:A` | Other mammals (negative, Vulpes excluded) | `data/raw/nonfox/XC{id}.mp3` |

Features:
- Handles pagination (`numPages` / `page`)
- Prepends `https:` to `//`-prefixed file URLs
- Skips already-downloaded files
- `time.sleep(0.5)` between requests
- `run_pipeline.sh` updated with optional Step 0 (requires `XC_KEY` env var)

In [18]:
# ── Step 12: Verify download_data.py exists & imports cleanly ───────────
import importlib, pathlib

project = pathlib.Path(PROJECT_ROOT)

# 12a – file exists
dl_file = project / "src" / "download_data.py"
assert dl_file.exists(), f"Missing: {dl_file}"
print(f"✓ {dl_file.relative_to(project)} exists ({dl_file.stat().st_size} bytes)")

# 12b – imports without error
spec = importlib.util.spec_from_file_location("download_data", str(dl_file))
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
assert hasattr(mod, "download_data"), "download_data() function not found"
assert hasattr(mod, "_query_xc"), "_query_xc() helper not found"
print("✓ download_data module imports cleanly")

# 12c – run_pipeline.sh now contains download step
sh = project / "run_pipeline.sh"
sh_text = sh.read_text()
assert "download_data" in sh_text, "run_pipeline.sh missing download step"
print("✓ run_pipeline.sh includes Step 0 (data download)")

# 12d – 01_data_exploration.ipynb updated
import json
nb01 = json.loads((project / "notebooks" / "01_data_exploration.ipynb").read_text())
nb01_text = "".join("".join(c["source"]) for c in nb01["cells"])
assert "download_data" in nb01_text, "01_data_exploration.ipynb missing download reference"
print(f"✓ 01_data_exploration.ipynb references download_data ({len(nb01['cells'])} cells)")

print("\n✅ Step 12 verified.")

✓ src/download_data.py exists (7208 bytes)
✓ download_data module imports cleanly
✓ run_pipeline.sh includes Step 0 (data download)
✓ 01_data_exploration.ipynb references download_data (11 cells)

✅ Step 12 verified.


## Step 13 — Training Set Confusion Matrices

Evaluate both models on the **training data** to check for overfitting.
If train and test metrics are close, the models generalise well.

In [20]:
# ── Step 13: Training-set confusion matrices ──────────────────────────
import os, sys, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score,
                             precision_score, recall_score, classification_report)
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.join(os.getcwd(), "fox_detection"))

from src.baseline_model import BaselineClassifier
from src.features import extract_mfcc_dataset
from src.cnn_model import FoxCNN
from src.dataset import FoxSpectrogramDataset

PROJECT = os.path.join(os.getcwd(), "fox_detection")
MANIFEST = os.path.join(PROJECT, "data", "manifest.csv")
FEATURE_DIR = os.path.join(PROJECT, "data", "features")
SPEC_DIR = os.path.join(PROJECT, "data", "spectrograms")
BASELINE_PATH = os.path.join(PROJECT, "models", "baseline", "model.pkl")
CNN_PATH = os.path.join(PROJECT, "models", "cnn", "best.pt")

# ── 1. Baseline SVM on training set ──────────────────────────────────
clf = BaselineClassifier.load(BASELINE_PATH)
X, y = extract_mfcc_dataset(MANIFEST, feature_dir=FEATURE_DIR)
X_train, _, y_train, _ = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
y_pred_bl = clf.pipeline.predict(X_train)

# ── 2. CNN on training set ───────────────────────────────────────────
dev = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
ckpt = torch.load(CNN_PATH, map_location=dev, weights_only=False)
model = FoxCNN(backbone=ckpt.get("backbone", "efficientnet_b0"), pretrained=False, num_classes=2)
model.load_state_dict(ckpt["model_state_dict"])
model.to(dev).eval()

train_ds = FoxSpectrogramDataset(MANIFEST, SPEC_DIR, split="train", img_size=(128, 128), augment=False)
loader = DataLoader(train_ds, batch_size=32, shuffle=False)
all_labels, all_preds = [], []
with torch.no_grad():
    for imgs, labs in loader:
        all_labels.extend(labs.tolist())
        all_preds.extend(model(imgs.to(dev)).argmax(dim=1).cpu().tolist())
y_true_cnn = np.array(all_labels)
y_pred_cnn = np.array(all_preds)

# ── 3. Plot side-by-side confusion matrices ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, title, yt, yp, cmap in [
    (axes[0], "SVM Baseline — Training Set", y_train, y_pred_bl, "Blues"),
    (axes[1], "CNN (EfficientNet-B0) — Training Set", y_true_cnn, y_pred_cnn, "Oranges"),
]:
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=["nonfox", "fox"],
                yticklabels=["nonfox", "fox"],
                ax=ax, annot_kws={"size": 18})
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("Actual", fontsize=12)
    ax.set_title(title, fontsize=13)

    acc = accuracy_score(yt, yp)
    f1 = f1_score(yt, yp, average="macro")
    ax.text(0.5, -0.15, f"Accuracy: {acc:.4f}  |  F1: {f1:.4f}",
            transform=ax.transAxes, ha="center", fontsize=11)

fig.suptitle("Training Set Evaluation (n = 12,347)", fontsize=15, y=1.02)
fig.tight_layout()
plt.show()

# ── 4. Print detailed reports ────────────────────────────────────────
print("=" * 60)
print("  SVM BASELINE — TRAINING SET")
print("=" * 60)
print(classification_report(y_train, y_pred_bl, target_names=["nonfox", "fox"]))

print("=" * 60)
print("  CNN (EfficientNet-B0) — TRAINING SET")
print("=" * 60)
print(classification_report(y_true_cnn, y_pred_cnn, target_names=["nonfox", "fox"]))

print("✅ Training-set evaluation complete.")

MFCC features: 100%|██████████| 17639/17639 [00:03<00:00, 4860.46it/s]


MFCC dataset: X (17639, 80), y (17639,)  (fox=5887, nonfox=11752)
  SVM BASELINE — TRAINING SET
              precision    recall  f1-score   support

      nonfox       1.00      0.99      1.00      8226
         fox       0.99      1.00      0.99      4121

    accuracy                           0.99     12347
   macro avg       0.99      0.99      0.99     12347
weighted avg       0.99      0.99      0.99     12347

  CNN (EfficientNet-B0) — TRAINING SET
              precision    recall  f1-score   support

      nonfox       1.00      1.00      1.00      8226
         fox       1.00      1.00      1.00      4121

    accuracy                           1.00     12347
   macro avg       1.00      1.00      1.00     12347
weighted avg       1.00      1.00      1.00     12347

✅ Training-set evaluation complete.


/var/folders/0m/8x59vmp96730r0vxr_ssqtrh0000gn/T/ipykernel_1121/4036987191.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
